In [27]:
import pandas as pd
import numpy as np

df = pd.read_csv("data.csv")
df.shape
X = df.drop(["id", "diagnosis"], axis=1)
y = df["diagnosis"]

print(X.shape, y.shape)

y.replace("M", 1, inplace=True)
y.replace("B", 0, inplace=True)


(569, 31) (569,)


0      1
1      1
2      1
3      1
4      1
      ..
564    1
565    1
566    1
567    1
568    0
Name: diagnosis, Length: 569, dtype: object

In [28]:
prior_1 = y.sum() / len(y)
prior_0 = 1 - prior_1

In [29]:
X_1 = X[y==1]
X_0 = X[y==0]

In [30]:
means_1 = {}
variances_1 = {}
means_0 = {}
variances_0 = {}

for col_name in X_1.columns:
    means_1[col_name] = X_1[col_name].mean()
    variances_1[col_name] = X_1[col_name].var()

for col_name in X_0.columns:
    means_0[col_name] = X_0[col_name].mean()
    variances_0[col_name] = X_0[col_name].var()

In [31]:
cov_1 = X_1.cov()
cov_0 = X_0.cov()

pooled_cov = prior_0 * cov_0 + prior_1 * cov_1

In [32]:
cov_inverse = np.linalg.inv(pooled_cov)

In [33]:
def lda_predict(x):
    diff_1 = x - pd.Series(means_1)
    diff_1 = diff_1.to_numpy()
    dist_1 = diff_1 @ cov_inverse @ diff_1.T

    score_1 = np.log(prior_1) - 0.5 * dist_1

    diff_0 = x - pd.Series(means_0)
    diff_0 = diff_0.to_numpy()
    dist_0 = diff_0 @ cov_inverse @ diff_0.T

    score_0 = np.log(prior_0) - 0.5 * dist_0

    if score_0 > score_1:
        return 0
    else:
        return 1

In [34]:
predictions = []
for index, person in X.iterrows():
    predictions.append(lda_predict(person))

In [35]:
from sklearn.metrics import classification_report

classification_report(y.astype(int), predictions)

/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

'              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       357\n           1       0.37      1.00      0.54       212\n\n    accuracy                           0.37       569\n   macro avg       0.19      0.50      0.27       569\nweighted avg       0.14      0.37      0.20       569\n'

In [36]:
cov_0_inverse = np.linalg.inv(cov_0)
cov_1_inverse = np.linalg.inv(cov_1)

In [37]:
cov_0_det = np.linalg.det(cov_0)
cov_1_det = np.linalg.det(cov_1)

/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2407: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2407: RuntimeWarning: invalid value encountered in det
  r = _umath_linalg.det(a, signature=signature)


In [38]:
def qda(x):
    diff_0 = x - pd.Series(means_0)
    diff_0 = diff_0.to_numpy()
    dist_0 = diff_0 @ cov_0_inverse @ diff_0.T

    score_0 = np.log(prior_0) - (0.5 * np.log(cov_0_det)) - (0.5 * dist_0)

    diff_1 = x - pd.Series(means_1)
    diff_1 = diff_1.to_numpy()
    dist_1 = diff_1 @ cov_1_inverse @ diff_1.T

    score_1 = np.log(prior_1) - (0.5 * np.log(cov_1_det)) - (0.5 * dist_1)

    if score_0 > score_1:
        return 0
    else:
        return 1

In [39]:
predictions = []
for index, x in X.iterrows():
    predictions.append(qda(x))

In [41]:
classification_report(y.astype(int), predictions)

/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jathin/.conda/envs/myenv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()}

'              precision    recall  f1-score   support\n\n           0       0.00      0.00      0.00       357\n           1       0.37      1.00      0.54       212\n\n    accuracy                           0.37       569\n   macro avg       0.19      0.50      0.27       569\nweighted avg       0.14      0.37      0.20       569\n'